# 🔬 Advanced LLM Activation Steering — Gaussian Concept Sampling (GCS)
## Concept: {EMOTION} (Full Evaluation Pipeline)

This notebook implements Advanced GCS (AGCS) for concept steering.

Key features:
- ✅ Concept difference direction: `μ = μ_pos − μ_neg` with pooled within-class σ
- ✅ Gaussian σ-ring sampling: `(k−1)σ < ‖x − μ‖ < kσ` (component-wise)
- ✅ Additive steering applied to the full sequence (fixed critical last-token-only bug)
- ✅ Chat template support for Instruct models
- ✅ Steering restricted to assistant tokens only
- ✅ Multi-seed evaluation with confidence intervals

In [ ]:
# 📦 Install dependencies
!pip install torch transformers openai python-dotenv numpy pandas matplotlib seaborn scikit-learn tqdm --quiet

In [ ]:
# 🔧 Imports & Setup
import os
import gc
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional, Tuple
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

# Global reproducibility
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

In [ ]:
# ⚙️ CONFIGURATION
EMOTION = "EVIL"
CONTROL_WORD = "Healthy"

class Config:
    STEERING_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
    STEER_LAYERS = list(range(10, 24))
    NUM_STEERING_LAYERS = 20
    BATCH_SIZE_HS = 4
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE = torch.float16
    CACHE_DIR = "./cache"
    RESULTS_DIR = "./results"
    GCS_NUM_SIGMA = 2.0       # k in formula: samples in ring ((k-1)σ, kσ)
    GCS_NUM_SAMPLES = 64      # vectors to sample per layer
    GCS_USE_DIFF = True       # use direction = μ_pos − μ_neg
    GCS_SHRINKAGE = 0.1       # shrinkage for pooled covariance

cfg = Config()
os.makedirs(cfg.CACHE_DIR, exist_ok=True)
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)

print(f"✅ Config OK. Device: {cfg.DEVICE}, model: {cfg.STEERING_MODEL}")

In [ ]:
# 📥 Load dataset
DATASET_PATH = f"./datasets/{EMOTION.lower()}_dataset.json"

print(f"📥 Loading dataset for '{EMOTION}' from {DATASET_PATH}...")
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

print(f"✅ Loaded {len(raw_dataset)} examples.")

def make_full_context(example, response_key):
    return f"Situation: {example['prompt']}\n\nResponse: {example[response_key]}"

pos_texts = [make_full_context(ex, EMOTION.lower()) for ex in raw_dataset]
neg_texts = [make_full_context(ex, "neutral") for ex in raw_dataset]

print(f"✅ Pos ({EMOTION}): {len(pos_texts)} | Neg ({CONTROL_WORD}): {len(neg_texts)}")

In [ ]:
# 📥 Load model
print(f"📥 Loading: {cfg.STEERING_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(cfg.STEERING_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# CRITICAL: left-padding ensures the last token is the real semantic end, not padding.
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    cfg.STEERING_MODEL,
    dtype=cfg.DTYPE,
    device_map="auto" if cfg.DEVICE == "cuda" else None,
    low_cpu_mem_usage=True,
)

print(f"✅ Model loaded. Hidden size: {model.config.hidden_size}, num layers: {model.config.num_hidden_layers}")
print(f"✅ Padding side: {tokenizer.padding_side}")

# Save original layers for reset
original_layers = [layer for layer in model.model.layers]

def reset_model(model, original_layers):
    for i, layer in enumerate(original_layers):
        model.model.layers[i] = layer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# 🔍 Extract hidden states (last-token activation)
def extract_hidden_states(
    texts: List[str],
    model,
    tokenizer,
    layers: List[int],
    batch_size: int = 4,
    max_length: int = 128
) -> Dict[int, torch.Tensor]:
    """Extract last-token activation at each requested layer."""
    all_states = {l: [] for l in layers}

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting hidden states"):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states

            for layer_idx in layers:
                last_token_states = hidden_states[layer_idx][:, -1, :].float().cpu()
                all_states[layer_idx].append(last_token_states)

            del inputs, outputs, hidden_states
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    for layer_idx in layers:
        all_states[layer_idx] = torch.cat(all_states[layer_idx], dim=0)
    return all_states

def clear_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
# Extract or load cached activations
pos_states_path = os.path.join(cfg.CACHE_DIR, "pos_states.pkl")
neg_states_path = os.path.join(cfg.CACHE_DIR, "neg_states.pkl")

if os.path.exists(pos_states_path) and os.path.exists(neg_states_path):
    print("📂 Loading previously cached activations...")
    with open(pos_states_path, "rb") as f:
        pos_states = pickle.load(f)
    with open(neg_states_path, "rb") as f:
        neg_states = pickle.load(f)
else:
    print("🔍 Extracting positive activations...")
    pos_states = extract_hidden_states(
        pos_texts, model, tokenizer, cfg.STEER_LAYERS, batch_size=cfg.BATCH_SIZE_HS
    )
    clear_gpu_memory()

    print("🔍 Extracting negative activations...")
    neg_states = extract_hidden_states(
        neg_texts, model, tokenizer, cfg.STEER_LAYERS, batch_size=cfg.BATCH_SIZE_HS
    )
    clear_gpu_memory()

    with open(pos_states_path, "wb") as f:
        pickle.dump(pos_states, f)
    with open(neg_states_path, "wb") as f:
        pickle.dump(neg_states, f)
    print("✅ Activations saved to cache.")

print(f"✅ Shapes: pos={pos_states[cfg.STEER_LAYERS[0]].shape}, neg={neg_states[cfg.STEER_LAYERS[0]].shape}")

In [ ]:
# ⚙️ Steering Layer (full-sequence, position-aware)
class SteeringLayer(nn.Module):
    """Additive steering applied to the full sequence starting from `start_pos`."""
    def __init__(
        self,
        target_layer: nn.Module,
        layer_idx: int,
        steering_vector: np.ndarray,
        strength: float = 0.1,
        h_ref_norm: float = 1.0,
    ):
        super().__init__()
        self.target_layer = target_layer
        self.layer_idx = layer_idx
        self.strength = strength
        self.h_ref_norm = h_ref_norm

        v = np.asarray(steering_vector, dtype=np.float32)
        v = v / (np.linalg.norm(v) + 1e-8)
        self.steering_vector_np = v
        self.start_pos = 0

    def __getattr__(self, name: str):
        return getattr(self.target_layer, name)

    def set_start_pos(self, pos: int):
        self.start_pos = int(pos)

    def forward(self, *args, **kwargs):
        original_output = self.target_layer(*args, **kwargs)
        if isinstance(original_output, tuple):
            hidden_states = original_output[0]
            rest = original_output[1:]
        else:
            hidden_states = original_output
            rest = None

        seq_len = hidden_states.shape[1]
        v = torch.tensor(self.steering_vector_np, dtype=hidden_states.dtype, device=hidden_states.device)
        delta = (self.strength * self.h_ref_norm) * v

        if seq_len == 1:
            hidden_states = hidden_states + delta.view(1, 1, -1)
        else:
            sp = min(self.start_pos, seq_len)
            if sp < seq_len:
                hidden_states = hidden_states.clone()
                hidden_states[:, sp:, :] = hidden_states[:, sp:, :] + delta.view(1, 1, -1)

        if rest is not None:
            return (hidden_states,) + rest
        return hidden_states


def apply_steering_to_model(
    model: nn.Module,
    steering_vectors: Dict[int, np.ndarray],
    strength: float,
    layers: List[int],
    h_ref_norms: Optional[Dict[int, float]] = None,
) -> nn.Module:
    for layer_idx in layers:
        if layer_idx in steering_vectors:
            original_layer = model.model.layers[layer_idx]
            h_ref = h_ref_norms.get(layer_idx, 1.0) if h_ref_norms else 1.0
            model.model.layers[layer_idx] = SteeringLayer(
                target_layer=original_layer,
                layer_idx=layer_idx,
                steering_vector=steering_vectors[layer_idx],
                strength=strength,
                h_ref_norm=h_ref,
            )
    return model


def set_steering_start_pos(model, layers: List[int], start_pos: int):
    """Set `start_pos` for each steering layer before `generate()`."""
    for layer_idx in layers:
        layer = model.model.layers[layer_idx]
        if isinstance(layer, SteeringLayer):
            layer.set_start_pos(start_pos)

print("✅ SteeringLayer (full-sequence, position-aware) defined")

In [ ]:
# 🎲 Gaussian Concept Sampling (GCS)
def _pooled_within_class_stats(
    X_pos: np.ndarray, X_neg: np.ndarray, shrinkage: float = 0.1
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (μ_pos, μ_neg, σ_pooled) with shrinkage."""
    mu_pos = X_pos.mean(axis=0)
    mu_neg = X_neg.mean(axis=0)
    Xc_pos = X_pos - mu_pos
    Xc_neg = X_neg - mu_neg
    n_pos, n_neg = X_pos.shape[0], X_neg.shape[0]
    var = (np.sum(Xc_pos**2, axis=0) + np.sum(Xc_neg**2, axis=0)) / max(n_pos + n_neg - 2, 1)
    mean_var = float(var.mean())
    var = (1.0 - shrinkage) * var + shrinkage * mean_var
    sigma = np.sqrt(np.maximum(var, 1e-12))
    return mu_pos, mu_neg, sigma


def gcs_sample_in_shell(
    mu: np.ndarray,
    sigma: np.ndarray,
    num_samples: int,
    k: float = 1.0,
    max_iter: int = 20,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    """Sample in σ-ring: reject points entirely inside the inner hypercube."""
    if rng is None:
        rng = np.random.default_rng()
    d = mu.shape[0]
    low_k = mu - k * sigma
    high_k = mu + k * sigma

    samples = rng.uniform(low=low_k, high=high_k, size=(num_samples, d))
    if k <= 1.0 + 1e-9:
        return samples

    low_inner = mu - (k - 1.0) * sigma
    high_inner = mu + (k - 1.0) * sigma
    inside_inner = np.all((samples >= low_inner) & (samples <= high_inner), axis=1)

    iter_count = 0
    while np.any(inside_inner) and iter_count < max_iter:
        n_bad = int(inside_inner.sum())
        samples[inside_inner] = rng.uniform(low=low_k, high=high_k, size=(n_bad, d))
        inside_inner = np.all((samples >= low_inner) & (samples <= high_inner), axis=1)
        iter_count += 1

    if np.any(inside_inner):
        bad_idx = np.where(inside_inner)[0]
        for i in bad_idx:
            axis = rng.integers(0, d)
            sign = rng.choice([-1.0, 1.0])
            samples[i, axis] = mu[axis] + sign * (k - 1e-3) * sigma[axis]
    return samples


def compute_gcs_steering(
    pos_states: Dict[int, torch.Tensor],
    neg_states: Dict[int, torch.Tensor],
    layers: List[int],
    num_sigma: float = 1.0,
    num_samples: int = 64,
    use_diff: bool = True,
    shrinkage: float = 0.1,
    seed: int = 42,
) -> Dict[int, np.ndarray]:
    """Compute mean steering vectors per layer using GCS."""
    rng = np.random.default_rng(seed)
    mean_vectors: Dict[int, np.ndarray] = {}

    for layer in tqdm(layers, desc="GCS sampling"):
        X_pos = pos_states[layer].cpu().numpy().astype(np.float32)
        X_neg = neg_states[layer].cpu().numpy().astype(np.float32)

        mu_pos, mu_neg, sigma = _pooled_within_class_stats(X_pos, X_neg, shrinkage)
        mu = (mu_pos - mu_neg) if use_diff else mu_pos

        samples = gcs_sample_in_shell(mu=mu, sigma=sigma, num_samples=num_samples, k=num_sigma, rng=rng)

        norms = np.linalg.norm(samples, axis=1, keepdims=True) + 1e-8
        samples_unit = samples / norms
        mean_dir = samples_unit.mean(axis=0)
        mean_dir = mean_dir / (np.linalg.norm(mean_dir) + 1e-8)

        proj_pos = X_pos @ mean_dir
        proj_neg = X_neg @ mean_dir
        margin = (proj_pos.mean() - proj_neg.mean()) / (proj_pos.std() + proj_neg.std() + 1e-8)

        mean_vectors[layer] = mean_dir
        print(f"  Layer {layer}: ||v̄||={np.linalg.norm(mean_dir):.4f}, margin={margin:+.3f}, samples={samples_unit.shape}")

    return mean_vectors


def compute_h_ref_norms(
    pos_states: Dict[int, torch.Tensor],
    neg_states: Dict[int, torch.Tensor],
    layers: List[int]
) -> Dict[int, float]:
    h_ref_norms = {}
    for layer in layers:
        H = torch.cat([pos_states[layer], neg_states[layer]], dim=0).cpu().numpy().astype(np.float32)
        norms = np.linalg.norm(H, axis=1)
        h_ref_norms[layer] = float(norms.mean())
    return h_ref_norms

In [ ]:
# Compute or load GCS vectors
gcs_file = os.path.join(cfg.CACHE_DIR, "gcs_data.pkl")

if os.path.exists(gcs_file) and os.path.getsize(gcs_file) > 0:
    print("📂 Loading GCS data from cache...")
    with open(gcs_file, 'rb') as f:
        data = pickle.load(f)
    steering_vectors = data['mean_vectors']
    h_ref_norms = data['h_ref_norms']
else:
    print(f"🧮 Running GCS (k={cfg.GCS_NUM_SIGMA}, n_samples={cfg.GCS_NUM_SAMPLES}, use_diff={cfg.GCS_USE_DIFF})...")
    steering_vectors = compute_gcs_steering(
        pos_states, neg_states,
        layers=cfg.STEER_LAYERS,
        num_sigma=cfg.GCS_NUM_SIGMA,
        num_samples=cfg.GCS_NUM_SAMPLES,
        use_diff=cfg.GCS_USE_DIFF,
        shrinkage=cfg.GCS_SHRINKAGE,
        seed=GLOBAL_SEED,
    )
    h_ref_norms = compute_h_ref_norms(pos_states, neg_states, cfg.STEER_LAYERS)
    print("📏 Reference activation norms per layer:")
    for l, n in h_ref_norms.items():
        print(f"  Layer {l}: ||h̄|| = {n:.2f}")

    with open(gcs_file, 'wb') as f:
        pickle.dump({'mean_vectors': steering_vectors, 'h_ref_norms': h_ref_norms}, f)
    print(f"✅ Saved {gcs_file}")

In [ ]:
# 📊 PCA Visualization
def visualize_pca_improved(pos_states, neg_states, layers):
    available_layers = sorted(pos_states.keys())
    selected_layers = [available_layers[0],
                       available_layers[len(available_layers)//3],
                       available_layers[2*len(available_layers)//3],
                       available_layers[-1]]

    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    separation_scores = []

    for idx, layer in enumerate(selected_layers):
        pos_data = pos_states[layer].cpu().numpy()
        neg_data = neg_states[layer].cpu().numpy()
        X = np.vstack([pos_data, neg_data])
        y = np.array([1]*len(pos_data) + [0]*len(neg_data))

        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X)

        silhouette = silhouette_score(X, y)
        lda_acc = LinearDiscriminantAnalysis(n_components=1).fit(X, y).score(X, y)

        separation_scores.append({
            'layer': layer, 'silhouette': silhouette,
            'lda_accuracy': lda_acc,
            'variance_explained': pca.explained_variance_ratio_[0]
        })

        axes[idx].scatter(X_pca[:len(pos_data), 0], X_pca[:len(pos_data), 1],
                          label=f'{EMOTION}', alpha=0.7, c='red')
        axes[idx].scatter(X_pca[len(pos_data):, 0], X_pca[len(pos_data):, 1],
                          label=f'{CONTROL_WORD}', alpha=0.7, c='blue')
        axes[idx].set_title(f'Layer {layer} (Sil: {silhouette:.3f}, LDA: {lda_acc:.3f})')
        axes[idx].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.RESULTS_DIR, "pca_layers.png"), dpi=120)
    plt.show()

    print("\n📊 PCA Separation Metrics:")
    print("="*70)
    for s in separation_scores:
        print(f"  Layer {s['layer']: >2}: silhouette={s['silhouette']:+.3f}, LDA-acc={s['lda_accuracy']:.3f}, var-exp={s['variance_explained']:.3f}")
    best = max(separation_scores, key=lambda x: x['silhouette'])
    print(f"\n✅ Best layer for separation: Layer {best['layer']} (silhouette={best['silhouette']:.3f})")
    return separation_scores

In [ ]:
# 📊 Probing Accuracy Evaluation
def evaluate_probing_accuracy_improved(pos_states, neg_states, layers):
    available_layers = sorted(pos_states.keys())
    accuracies = {}

    for layer in available_layers:
        X = torch.cat([pos_states[layer], neg_states[layer]], dim=0).cpu().numpy()
        y = np.array([1]*len(pos_states[layer]) + [0]*len(neg_states[layer]))
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=GLOBAL_SEED, stratify=y
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        clf = LogisticRegression(max_iter=1000, fit_intercept=True, C=1.0)
        clf.fit(X_train_s, y_train)
        y_pred = clf.predict(X_test_s)
        accuracies[layer] = {'test_acc': accuracy_score(y_test, y_pred),
                             'f1': f1_score(y_test, y_pred)}

    print("\n📊 Probing Classifier (held-out test):")
    print("="*60)
    print(f"{'layer': <8}{'test_acc': <12}{'f1': <12}")
    for l, m in accuracies.items():
        print(f"{l: <8}{m['test_acc']: <12.4f}{m['f1']: <12.4f}")
    best = max(accuracies, key=lambda x: accuracies[x]['test_acc'])
    print(f"\n✅ Best layer by test accuracy: L{best} ({accuracies[best]['test_acc']:.4f})")
    pd.DataFrame(accuracies).T.to_csv(os.path.join(cfg.RESULTS_DIR, "probing_accuracy.csv"))
    return accuracies

In [ ]:
# 📊 Run visualization & probing
print("🔬 Visualization + probing...")
sep_scores = visualize_pca_improved(pos_states, neg_states, cfg.STEER_LAYERS)
probing_accuracies = evaluate_probing_accuracy_improved(pos_states, neg_states, cfg.STEER_LAYERS)

# Free memory
del pos_states, neg_states
clear_gpu_memory()

# Select top-N steering layers from mid-range (12-22)
ranked = sorted(probing_accuracies.items(), key=lambda x: x[1]['test_acc'], reverse=True)
mid_range = [l for l, _ in ranked if 12 <= l <= 22]
chosen = mid_range[:cfg.NUM_STEERING_LAYERS] if len(mid_range) >= cfg.NUM_STEERING_LAYERS else [l for l, _ in ranked[:cfg.NUM_STEERING_LAYERS]]

cfg.STEER_LAYERS = sorted(chosen)
print(f"🎯 Chosen steering layers (top-{cfg.NUM_STEERING_LAYERS} probing within mid range): {cfg.STEER_LAYERS}")

In [ ]:
clear_gpu_memory()
print("✅ GPU memory cleared")

In [ ]:
# 🧪 Steering Effect Testing Loop
import csv

TEST_PROMPTS = [
    "Tell a short story about a person's choice.",
    "Write a movie review in 2-3 sentences.",
    "What do you think about the importance of moral principles?",
    "What would you do if you had one free day with no obligations?",
    "If you could instantly learn one skill without effort, what would it be and why?",
    "Invent a new holiday and describe how people celebrate it."
]

STEERING_STRENGTHS = [round(i * 0.02, 3) for i in range(0, 14)]  # 0.0 ... 0.26
N_SEEDS = 1
RESULT_CSV = os.path.join(cfg.RESULTS_DIR, "generation_results.csv")

def build_chat_inputs(prompt: str, tokenizer, device):
    """Format chat template and return (input_ids, attention_mask, start_pos)."""
    messages = [{"role": "user", "content": prompt}]
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
    )
    if hasattr(encoded, "to"):
        encoded = encoded.to(device)
        input_ids = encoded["input_ids"]
        attention_mask = encoded.get("attention_mask", torch.ones_like(input_ids))
    elif isinstance(encoded, dict):
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded.get("attention_mask", torch.ones_like(input_ids)).to(device)
    else:
        raise TypeError(f"Unexpected output from apply_chat_template: {type(encoded)}")

    start_pos = input_ids.shape[1]
    return input_ids, attention_mask, start_pos

def generate_with_steering(
    prompt: str, model, tokenizer,
    steering_vectors=None, strength: float = 0.0,
    layers: List[int] = None, h_ref_norms=None,
    max_new_tokens: int = 100, temperature: float = 0.7, seed: int = 0
) -> str:
    reset_model(model, original_layers)
    clear_gpu_memory()
    torch.manual_seed(seed)
    np.random.seed(seed)

    input_ids, attention_mask, start_pos = build_chat_inputs(prompt, tokenizer, model.device)

    if steering_vectors and strength != 0.0 and layers:
        apply_steering_to_model(model, steering_vectors, strength, layers, h_ref_norms)
        set_steering_start_pos(model, layers, start_pos)

    with torch.no_grad():
        outputs = model.generate(
            input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=True, pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    gen_tokens = outputs[0, input_ids.shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

def get_existing_keys(csv_path: str):
    existing = set()
    if os.path.exists(csv_path):
        with open(csv_path, 'r', newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                existing.add((row['prompt'], row['strength'], row['seed']))
    return existing

def save_results_to_csv(results, csv_path, mode='a'):
    file_exists = os.path.exists(csv_path)
    with open(csv_path, mode, newline='', encoding='utf-8') as f:
        fieldnames = ['prompt_id', 'prompt', 'strength', 'seed', 'response', 'response_length']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists or mode == 'w':
            writer.writeheader()
        writer.writerows(results)

# Main generation loop
existing = get_existing_keys(RESULT_CSV)
total_runs = 0

for prompt_idx, prompt in enumerate(TEST_PROMPTS, start=1):
    print(f"\n📝 Prompt {prompt_idx}: '{prompt[:60]}...'")
    prompt_results = []

    for strength in tqdm(STEERING_STRENGTHS, desc=f"P{prompt_idx} strengths"):
        for seed in range(N_SEEDS):
            key = (prompt, str(strength), str(seed))
            if key in existing:
                continue
            response = generate_with_steering(
                prompt, model, tokenizer, steering_vectors=steering_vectors,
                strength=strength, layers=cfg.STEER_LAYERS, h_ref_norms=h_ref_norms,
                max_new_tokens=80, seed=seed
            )
            prompt_results.append({
                'prompt_id': prompt_idx, 'prompt': prompt, 'strength': strength,
                'seed': seed, 'response': response, 'response_length': len(response.split())
            })
            total_runs += 1

    if prompt_results:
        save_results_to_csv(prompt_results, RESULT_CSV, mode='a')
        for r in prompt_results:
            if r['strength'] in (0.0, 0.2, 0.4) and r['seed'] == 0:
                print(f"  s={r['strength']:.2f}: {r['response'][:120]}")

print(f"\n✅ Done. New generations: {total_runs}. Results → {RESULT_CSV}")